In [2]:
%load_ext dotenv
%dotenv
import os
MODEL = 'gemini-3.1-flash-lite'

In [3]:
import gradio as gr
import google.generativeai as genai

genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

# 2. Define the Chat Function
def chat_with_gemini(message, history):
    """
    This function takes the current message and the chat history from Gradio,
    formats it for the Gemini API, and returns the model's response.
    """
    model = genai.GenerativeModel(MODEL)
    
    # Convert Gradio's history format (list of [user_text, bot_text] pairs) 
    # into the format expected by the Gemini API
    formatted_history = []
    for user_msg, bot_msg in history:
        formatted_history.append({"role": "user", "parts": [user_msg]})
        formatted_history.append({"role": "model", "parts": [bot_msg]})
        
    # Initialize the chat session with the formatted history
    chat = model.start_chat(history=formatted_history)
    
    # Send the user's new message and get the response
    try:
        response = chat.send_message(message)
        return response.text
    except Exception as e:
        return f"An error occurred: {str(e)}"

# 3. Create the Gradio Chat Interface
demo = gr.ChatInterface(
    fn=chat_with_gemini,
    title="Gemini Chat UI",
    description="A simple chat interface powered by Google's Gemini model and Gradio.",
    examples=["Hello!", "What are the main principles of the Model Context Protocol?", "Explain unsupervised learning to me."]
)

# 4. Launch the App
# In a Jupyter Notebook, this will render the UI directly below the cell.
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# 2. Define the Streaming Chat Function
def chat_with_gemini_stream(message, history):
    model = genai.GenerativeModel(MODEL)
    
    # Map Gradio history to Gemini format
    formatted_history = []
    for user_msg, bot_msg in history:
        formatted_history.append({"role": "user", "parts": [user_msg]})
        formatted_history.append({"role": "model", "parts": [bot_msg]})
        
    chat = model.start_chat(history=formatted_history)
    
    try:
        # Enable streaming by adding stream=True
        response = chat.send_message(message, stream=True)
        
        # Accumulate the text and yield it chunk by chunk
        partial_message = ""
        for chunk in response:
            partial_message += chunk.text
            # 'yield' hands the partial text back to Gradio immediately
            yield partial_message 
            
    except Exception as e:
        yield f"An error occurred: {str(e)}"

# 3. Create and launch the Gradio Interface
demo = gr.ChatInterface(
    fn=chat_with_gemini_stream, 
    title="Gemini Streaming Chat UI",
    description="Now with real-time text streaming!",
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
